# PixelDrop — Pandas Project
## Notebook 02: Clean the Data

**Goal:** Clean all 5 DataFrames and prepare them for analysis.

**Cleaning tasks:**
- Remove duplicates
- Fix date formats
- Standardise text columns (segment, state, status, etc.)
- Fix numeric columns (prices, quantities, discounts)
- Handle missing values
- Create calculated columns (line_total, year, month, etc.)

## Step 1: Import Libraries & Load Data

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

PATH = r'C:\Users\\איתי\Desktop\claude\data\raw\\'

customers   = pd.read_csv(PATH + 'customers.csv')
products    = pd.read_csv(PATH + 'products.csv')
orders      = pd.read_csv(PATH + 'orders.csv')
order_items = pd.read_csv(PATH + 'order_items.csv')
returns     = pd.read_csv(PATH + 'returns.csv')

print('All files loaded!')

All files loaded!


## Step 2: Remove Duplicates
Check and remove duplicate rows across all tables.
Focus on `orders` — we know it had duplicate order_ids.

In [2]:
# Check duplicates before
print('Duplicates before cleaning:')
for name, df in [('customers', customers), ('orders', orders), ('order_items', order_items)]:
    print(f'  {name}: {df.duplicated().sum()} duplicate rows')

# Remove duplicate order_ids from orders (keep first occurrence)
orders_before = len(orders)
orders = orders.drop_duplicates(subset='order_id', keep='first')
print(f'\nOrders: removed {orders_before - len(orders)} duplicate rows')
print(f'Orders remaining: {len(orders)}')

Duplicates before cleaning:
  customers: 0 duplicate rows
  orders: 0 duplicate rows
  order_items: 0 duplicate rows

Orders: removed 315 duplicate rows
Orders remaining: 11685


## Step 3: Fix Date Columns
`pd.to_datetime()` handles multiple formats automatically.
We set `errors='coerce'` so bad dates become NaT (null) instead of crashing.

In [3]:
# Fix dates in all 3 tables
customers['signup_date'] = pd.to_datetime(customers['signup_date'], errors='coerce', dayfirst=False)
orders['order_date']     = pd.to_datetime(orders['order_date'],     errors='coerce', dayfirst=False)
returns['return_date']   = pd.to_datetime(returns['return_date'],   errors='coerce', dayfirst=False)

print('Date columns fixed!')
print(f'Customers signup_date type: {customers["signup_date"].dtype}')
print(f'Orders order_date type:     {orders["order_date"].dtype}')
print(f'Returns return_date type:   {returns["return_date"].dtype}')

# Check for any dates that couldn't be parsed
print(f'\nNull dates after conversion:')
print(f'  signup_date: {customers["signup_date"].isnull().sum()}')
print(f'  order_date:  {orders["order_date"].isnull().sum()}')
print(f'  return_date: {returns["return_date"].isnull().sum()}')

Date columns fixed!
Customers signup_date type: datetime64[ns]
Orders order_date type:     datetime64[ns]
Returns return_date type:   datetime64[ns]

Null dates after conversion:
  signup_date: 3602
  order_date:  9363
  return_date: 1194


## Step 4: Create Date Helper Columns
Extract year, month, quarter, day_of_week from order_date.
These will be used heavily in analysis.

In [4]:
orders['year']        = orders['order_date'].dt.year
orders['month']       = orders['order_date'].dt.month
orders['quarter']     = orders['order_date'].dt.quarter
orders['day_of_week'] = orders['order_date'].dt.day_name()
orders['week'] = orders['order_date'].dt.isocalendar().week.astype('Int64')

print('Date helper columns created!')
print(orders[['order_id', 'order_date', 'year', 'month', 'quarter', 'day_of_week']].head())

Date helper columns created!
    order_id order_date    year  month  quarter day_of_week
0  ORD-10000 2024-10-16  2024.0   10.0      4.0   Wednesday
1  ORD-10001        NaT     NaN    NaN      NaN         NaN
2      10002        NaT     NaN    NaN      NaN         NaN
3  ORD-10003        NaT     NaN    NaN      NaN         NaN
4  ORD-10004        NaT     NaN    NaN      NaN         NaN


## Step 5: Standardise Text Columns
Map all dirty values to canonical clean values using a dictionary.
`.map()` replaces values, `.str.strip().str.title()` fixes casing.

In [5]:
# ── CUSTOMERS: segment ────────────────────────────────────────
segment_map = {
    'occasional': 'Occasional', 'occ': 'Occasional', 'OCCASIONAL': 'Occasional',
    'new': 'New', 'n': 'New', 'NEW': 'New', 'New': 'New',
    'loyal': 'Loyal', 'lyl': 'Loyal', 'LOYAL': 'Loyal', 'Loyal': 'Loyal',
    'vip': 'VIP', 'v.i.p': 'VIP', 'VIP': 'VIP', 'Vip': 'VIP',
}
customers['segment'] = customers['segment'].str.strip().map(
    lambda x: segment_map.get(str(x).lower().strip(), x) if pd.notna(x) else x
)
print('Segment values after cleaning:')
print(customers['segment'].value_counts())

Segment values after cleaning:
segment
Occasional    1837
Loyal         1107
New           1094
VIP            462
Name: count, dtype: int64


In [6]:
# ── CUSTOMERS: state ─────────────────────────────────────────
state_map = {
    'florida': 'FL', 'fla.': 'FL', 'fl': 'FL',
    'new york': 'NY', 'n.y': 'NY', 'ny': 'NY',
    'texas': 'TX', 'tx': 'TX', 'tex.': 'TX',
    'california': 'CA', 'calif.': 'CA', 'ca': 'CA',
    'illinois': 'IL', 'ill.': 'IL', 'il': 'IL',
    'washington': 'WA', 'wash.': 'WA', 'wa': 'WA',
    'colorado': 'CO', 'colo.': 'CO', 'co': 'CO',
    'georgia': 'GA', 'ga.': 'GA', 'ga': 'GA',
    'arizona': 'AZ', 'ariz.': 'AZ', 'az': 'AZ',
    'north carolina': 'NC', 'n.c': 'NC', 'nc': 'NC',
    'ohio': 'OH', 'oh': 'OH',
    'michigan': 'MI', 'mich.': 'MI', 'mi': 'MI',
    'new jersey': 'NJ', 'n.j': 'NJ', 'nj': 'NJ',
    'virginia': 'VA', 'va.': 'VA', 'va': 'VA',
    'massachusetts': 'MA', 'mass.': 'MA', 'ma': 'MA',
}
customers['state'] = customers['state'].str.strip().str.lower().map(
    lambda x: state_map.get(x, x.upper() if pd.notna(x) else x)
)
orders['shipping_state'] = orders['shipping_state'].str.strip().str.lower().map(
    lambda x: state_map.get(x, x.upper() if pd.notna(x) else x)
)
print('States cleaned!')
print(customers['state'].value_counts())

States cleaned!
state
NY    330
VA    321
NC    319
FL    318
GA    315
WA    307
CA    300
TX    300
OH    293
AZ    293
MI    286
IL    286
MA    282
CO    276
NJ    274
Name: count, dtype: int64


In [7]:
# ── CUSTOMERS: customer_name proper case ──────────────────────
customers['customer_name'] = customers['customer_name'].str.strip().str.title()

# ── ORDERS: order_status ──────────────────────────────────────
status_map = {
    'completed': 'Completed', 'complete': 'Completed', 'done': 'Completed',
    'refunded': 'Refunded', 'refund': 'Refunded',
    'pending': 'Pending', 'in progress': 'Pending',
    'cancelled': 'Cancelled', 'canceled': 'Cancelled',
}
orders['order_status'] = orders['order_status'].str.strip().str.lower().map(
    lambda x: status_map.get(x, x) if pd.notna(x) else x
)
print('Order status values:')
print(orders['order_status'].value_counts())

Order status values:
order_status
Completed    7696
Pending      1737
Cancelled    1149
Refunded     1103
Name: count, dtype: int64


In [8]:
# ── ORDERS: payment_method ────────────────────────────────────
payment_map = {
    'paypal': 'PayPal', 'pp': 'PayPal', 'pay pal': 'PayPal',
    'credit card': 'Credit Card', 'cc': 'Credit Card', 'credit': 'Credit Card', 'creditcard': 'Credit Card',
    'apple pay': 'Apple Pay', 'applepay': 'Apple Pay',
    'debit card': 'Debit Card', 'debit': 'Debit Card',
    'shop pay': 'Shop Pay', 'shoppay': 'Shop Pay',
}
orders['payment_method'] = orders['payment_method'].str.strip().str.lower().map(
    lambda x: payment_map.get(x, x) if pd.notna(x) else x
)
print('Payment methods:')
print(orders['payment_method'].value_counts())

Payment methods:
payment_method
Credit Card    2417
PayPal         2399
Debit Card     2341
Shop Pay       2264
Apple Pay      2264
Name: count, dtype: int64


In [9]:
# ── RETURNS: return_status ────────────────────────────────────
ret_status_map = {
    'approved': 'Approved', 'accepted': 'Approved',
    'pending': 'Pending', 'under review': 'Pending',
    'rejected': 'Rejected', 'denied': 'Rejected',
}
returns['return_status'] = returns['return_status'].str.strip().str.lower().map(
    lambda x: ret_status_map.get(x, x) if pd.notna(x) else x
)

# ── RETURNS: reason ───────────────────────────────────────────
reason_map = {
    'wrong size': 'Wrong Size', 'size issue': 'Wrong Size', 'wrong size ordered': 'Wrong Size',
    'defective': 'Defective', 'damaged': 'Defective', 'faulty': 'Defective',
    'changed mind': 'Changed Mind', 'no longer needed': 'Changed Mind',
    'dont want': 'Changed Mind', "don't want": 'Changed Mind',
    'wrong item': 'Wrong Item', 'incorrect item sent': 'Wrong Item',
    'not as described': 'Not As Described', 'misleading listing': 'Not As Described',
}
returns['reason'] = returns['reason'].str.strip().str.lower().map(
    lambda x: reason_map.get(x, x) if pd.notna(x) else x
)
print('Returns cleaned!')
print(returns['reason'].value_counts())

Returns cleaned!
reason
Wrong Item          321
Wrong Size          314
Defective           300
Not As Described    292
Changed Mind        273
Name: count, dtype: int64


In [10]:
# ── PRODUCTS: product_name & category ────────────────────────
product_name_map = {
    'air hoodie': 'Air Hoodie', 'bucket hat': 'Bucket Hat',
    'canvas tote': 'Canvas Tote', 'cargo pants': 'Cargo Pants',
    'chain neklace': 'Chain Necklace', 'classic t': 'Classic Tee',
    'drrop tee': 'Drop Tee', 'graphic tee vol2': 'Graphic Tee Vol2',
    'hightop force': 'High Top Force', 'jogger slim': 'Jogger Slim',
    'pixel cap snapback': 'PixelCap Snapback', 'pixelcap snapback': 'PixelCap Snapback', 'puffa jacket': 'Puffer Jacket',
    'retro runner sneaker': 'Retro Runner Sneaker', 'slidepro': 'Slide Pro',
    'zip hoodie': 'Zip Hoodie',
}
products['product_name'] = products['product_name'].str.strip().str.lower().map(
    lambda x: product_name_map.get(x, x.title()) if pd.notna(x) else x
)

category_map = {
    'hoodies': 'Hoodies', 'hoodie': 'Hoodies',
    'shoes': 'Sneakers', 'sneakers': 'Sneakers',
    'tees': 'T-Shirts', 't-shirts': 'T-Shirts', 'tshirts': 'T-Shirts',
    'bottoms': 'Bottoms', 'bottom': 'Bottoms',
    'accessories': 'Accessories', 'acc': 'Accessories',
    'caps': 'Caps', 'hats': 'Caps',
}
products['category'] = products['category'].str.strip().str.lower().map(
    lambda x: category_map.get(x, x.title()) if pd.notna(x) else x
)
print('Products cleaned!')
print(products[['product_name', 'category']])

Products cleaned!
            product_name     category
0             Air Hoodie      Hoodies
1            Classic Tee     T-Shirts
2               Drop Tee     T-Shirts
3            Cargo Pants      Bottoms
4            Jogger Slim      Bottoms
5      PixelCap Snapback         Caps
6             Bucket Hat         Caps
7   Retro Runner Sneaker     Sneakers
8         High Top Force     Sneakers
9              Slide Pro     Sneakers
10        Chain Necklace  Accessories
11           Canvas Tote  Accessories
12         Puffer Jacket      Hoodies
13            Zip Hoodie      Hoodies
14      Graphic Tee Vol2     T-Shirts


## Step 6: Fix Numeric Columns
Clean prices (strip $, fix commas), quantities, discounts.

In [11]:
# ── unit_price: strip $ and replace , with . ──────────────────
order_items['unit_price'] = (
    order_items['unit_price']
    .astype(str)
    .str.replace('$', '', regex=False)
    .str.replace(',', '.', regex=False)
    .str.strip()
)
order_items['unit_price'] = pd.to_numeric(order_items['unit_price'], errors='coerce')

# ── quantity: convert to numeric ─────────────────────────────
order_items['quantity'] = pd.to_numeric(order_items['quantity'], errors='coerce')

# ── discount: convert to numeric ─────────────────────────────
order_items['discount_%'] = pd.to_numeric(order_items['discount_%'], errors='coerce')

print('Numeric columns converted!')
print(order_items[['unit_price', 'quantity', 'discount_%']].dtypes)

Numeric columns converted!
unit_price    float64
quantity      float64
discount_%    float64
dtype: object


In [12]:
# ── Remove bad rows ───────────────────────────────────────────
before = len(order_items)

# Remove bad quantities
order_items = order_items[order_items['quantity'] > 0]

# Remove impossible discounts
order_items = order_items[order_items['discount_%'] <= 100]

# Fill null discounts with 0
order_items['discount_%'] = order_items['discount_%'].fillna(0)

after = len(order_items)
print(f'Removed {before - after} bad rows from order_items')
print(f'order_items remaining: {after}')

Removed 4677 bad rows from order_items
order_items remaining: 24360


In [13]:
# ── Fix shipping_days in orders ───────────────────────────────
orders['shipping_days'] = pd.to_numeric(orders['shipping_days'], errors='coerce')

# Remove impossible shipping days
orders.loc[orders['shipping_days'] < 0, 'shipping_days'] = np.nan
orders.loc[orders['shipping_days'] == 999, 'shipping_days'] = np.nan

print(f'Shipping days cleaned!')
print(f'Null shipping days: {orders["shipping_days"].isnull().sum()}')

Shipping days cleaned!
Null shipping days: 483


In [14]:
# ── Fix numeric columns in products ──────────────────────────
products['cost_price']   = pd.to_numeric(products['cost_price'],   errors='coerce')
products['retail_price'] = pd.to_numeric(products['retail_price'], errors='coerce')
products['stock_qty']    = pd.to_numeric(products['stock_qty'],    errors='coerce')

# Fill missing cost_price for Puffer Jacket
products.loc[products['product_name'] == 'Puffer Jacket', 'cost_price'] = 80.00

print('Products numeric columns fixed!')
print(products[['product_name', 'cost_price', 'retail_price']])

Products numeric columns fixed!
            product_name  cost_price  retail_price
0             Air Hoodie        45.0         89.99
1            Classic Tee        10.0         29.99
2               Drop Tee        12.0         34.99
3            Cargo Pants        52.0        119.99
4            Jogger Slim        33.0         79.99
5      PixelCap Snapback        18.0         44.99
6             Bucket Hat        15.0         39.99
7   Retro Runner Sneaker        85.0        179.99
8         High Top Force       100.0        219.99
9              Slide Pro        24.0         59.99
10        Chain Necklace         8.0         24.99
11           Canvas Tote         6.0         19.99
12         Puffer Jacket        80.0        189.99
13            Zip Hoodie        42.0         99.99
14      Graphic Tee Vol2        11.0         32.99


## Step 7: Validate Emails
Use regex to flag invalid emails — like we did in Script 06.

In [15]:
email_pattern = r'^[^@]+@[^@]+\.[^@]+$'
customers['is_valid_email'] = customers['email'].str.match(email_pattern, na=False).astype(int)

print(f'Valid emails:   {customers["is_valid_email"].sum()}')
print(f'Invalid emails: {(customers["is_valid_email"] == 0).sum()}')

Valid emails:   4349
Invalid emails: 151


In [16]:
# Flag duplicate emails
customers['is_duplicate_email'] = customers.duplicated(subset='email', keep=False).astype(int)
print(f'Duplicate emails: {customers["is_duplicate_email"].sum()}')

Duplicate emails: 725


## Step 8: Create the Main Analysis DataFrame
Merge all tables together — like `vw_order_summary` in SQL.
This is the main DataFrame we'll use for all analysis.

In [17]:
# Filter to clean customers only
clean_customers = customers[
    (customers['is_valid_email'] == 1) &
    (customers['is_duplicate_email'] == 0) &
    (customers['customer_name'].notna())
]

# Filter to valid orders (those that have matching items)
valid_order_ids = order_items['order_id'].unique()
clean_orders = orders[orders['order_id'].isin(valid_order_ids)]

print(f'Clean customers: {len(clean_customers)}')
print(f'Clean orders:    {len(clean_orders)}')

Clean customers: 3618
Clean orders:    11106


In [18]:
# Merge everything together
order_summary = (
    clean_orders
    .merge(order_items,    on='order_id',   how='left')
    .merge(products,       on='product_id', how='left')
    .merge(clean_customers, on='customer_id', how='left')
)

# Calculate line_total
order_summary['line_total'] = (
    order_summary['quantity'] *
    order_summary['unit_price'] *
    (1 - order_summary['discount_%'] / 100)
).round(2)

print(f'order_summary shape: {order_summary.shape}')
print(f'Columns: {list(order_summary.columns)}')

order_summary shape: (24360, 33)
Columns: ['order_id', 'order_date', 'customer_id', 'shipping_state', 'payment_method', 'order_status', 'shipping_days', 'notes', 'year', 'month', 'quarter', 'day_of_week', 'week', 'item_id', 'product_id', 'quantity', 'unit_price', 'discount_%', 'product_name', 'category', 'cost_price', 'retail_price', 'stock_qty', 'description', 'customer_name', 'email', 'phone', 'state', 'signup_date', 'segment', 'is_valid_email', 'is_duplicate_email', 'line_total']


In [19]:
# Preview the merged DataFrame
display(order_summary.head())

,order_id,order_date,customer_id,shipping_state,payment_method,order_status,shipping_days,notes,year,month,...,description,customer_name,email,phone,state,signup_date,segment,is_valid_email,is_duplicate_email,line_total
0,ORD-10000,2024-10-16,CUST-3992,NJ,Shop Pay,Completed,5.0,NaN,2024.0,10.0,...,Premium Cargo Pants from PixelDrop.,Aiden Thomas,aiden_thomas@gmail.com,(070) 206-6579,AZ,NaT,VIP,1.0,0.0,408.00
1,ORD-10000,2024-10-16,CUST-3992,NJ,Shop Pay,Completed,5.0,NaN,2024.0,10.0,...,PixelDrop Zip Hoodie — limited drop.,Aiden Thomas,aiden_thomas@gmail.com,(070) 206-6579,AZ,NaT,VIP,1.0,0.0,159.98
2,ORD-10001,NaT,CUST-1243,GA,Shop Pay,Refunded,10.0,urgent,NaN,NaN,...,Premium Cargo Pants from PixelDrop.,Sebastian Gonzalez,gonzalezs@yahoo.com,+14690909812,FL,NaT,Loyal,1.0,0.0,191.98
3,ORD-10001,NaT,CUST-1243,GA,Shop Pay,Refunded,10.0,urgent,NaN,NaN,...,Premium Classic Tee from PixelDrop.,Sebastian Gonzalez,gonzalezs@yahoo.com,+14690909812,FL,NaT,Loyal,1.0,0.0,25.50
4,ORD-10001,NaT,CUST-1243,GA,Shop Pay,Refunded,10.0,urgent,NaN,NaN,...,NaN,Sebastian Gonzalez,gonzalezs@yahoo.com,+14690909812,FL,NaT,Loyal,1.0,0.0,34.99


## Step 9: Save Clean DataFrames
Save cleaned versions to CSV for use in other notebooks.

In [20]:
OUTPUT = r'C:\Users\\איתי\Desktop\claude\data\clean\\'
import os
os.makedirs(OUTPUT, exist_ok=True)

order_summary.to_csv(OUTPUT + 'order_summary.csv', index=False)
clean_customers.to_csv(OUTPUT + 'customers_clean.csv', index=False)
products.to_csv(OUTPUT + 'products_clean.csv', index=False)
returns.to_csv(OUTPUT + 'returns_clean.csv', index=False)

print('Clean files saved to:', OUTPUT)

Clean files saved to: C:\Users\\איתי\Desktop\claude\\clean\\


## Summary — Cleaning Complete!

| Step | What we did |
|------|-------------|
| Duplicates | Removed duplicate order_ids |
| Dates | Converted all date columns to datetime |
| Date helpers | Added year, month, quarter, day_of_week |
| Segment | Standardised to 4 values: New, Occasional, Loyal, VIP |
| State | Standardised to 2-letter codes |
| Order status | Standardised to 4 values |
| Payment method | Standardised to 5 values |
| Return reason | Standardised to 5 values |
| Prices | Stripped $, fixed commas, converted to float |
| Quantities | Converted to numeric, removed bad rows |
| Discounts | Converted to numeric, removed > 100%, filled nulls with 0 |
| Emails | Validated with regex, flagged duplicates |
| Merged | Created `order_summary` — main analysis DataFrame |

**Next:** Notebook 03 — Product Analysis!